# AI Newsroom Studio — Pipeline Notebook
10-agent pipeline: HackerNews trends → fact-check → script → video → YouTube Shorts.

**Agents built:** Agent 1 (Trend Hunter) · Agent 2 (Context Researcher) · Agent 3 (Fact Checker)

---
## Setup: Shared State

In [ ]:
from typing_extensions import TypedDict
class NewsroomState(TypedDict):
    stories: dict   # per-story (Agents 1-4)
    script:  dict   # NEW top-level key (Agent 5 adds this)
    # script = {
    #   "full_text":    str,   # complete script
    #   "word_count":   int,   # verified count
    #   "est_duration": str,   # "74s"
    #   "sections":     dict,  # parsed labels
    #   "stories_used": list,  # [1, 2, 3] selection ranks
    #   "attempt":      int,   # 1 or 2 (audit trail)
    # }

---
## Agent 1: Trend Fetcher
Fetches top HackerNews stories and scores them by velocity.

**Velocity** = (upvotes + comments×2) / age_hrs — rewards recent engagement.

In [ ]:
from agents.agent1 import fetch_trends
import re

def make_story_id(title: str) -> str:
    """Slugify title to a stable dict key. e.g. 'Melody India Italy' → 'melody-india-italy'"""
    return re.sub(r'[^a-z0-9]+', '-', title.lower()).strip('-')

# AGENT 1 NODE
def trend_hunter_node(state: NewsroomState) -> dict:
    trends = fetch_trends()
    stories = {make_story_id(t["title"]): t for t in trends}
    return {"stories": stories}

In [ ]:
# Run Agent 1
call = trend_hunter_node(NewsroomState(stories={}))
print(f"Stories fetched: {len(call['stories'])}")
for sid, s in call['stories'].items():
    print(f"  {s['velocity']:>6.1f} vel | {sid}")

In [ ]:
# ── INSPECT after Agent 1 ────────────────────────────────────────────
# Agent 1 fields: title, url, source, category, engagement, velocity
from urllib.parse import urlparse

print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

---
## Agent 2: Context Researcher
For each story:
1. Fetches article content (3-tier: trafilatura → Jina → Tavily)
2. Gathers background (DDG news + Wikipedia)
3. Synthesises one tight background paragraph (qwen2.5:3b, content-anchored)

**Key design:**  rejects page chrome before accepting fetched text.
Backgrounds are honest-empty when DDG has no coverage (GitHub/arXiv/new tools — see KNOWN_ISSUES.md).

In [ ]:
import ollama, subprocess, time

# Health-check: Ollama must be running for Agent 2 synthesis
try:
    ollama.list()
    print("✅ Ollama already running")
except Exception:
    try:
        subprocess.Popen(["ollama", "serve"])
        time.sleep(3)
        ollama.list()
        print("✅ Ollama started")
    except Exception:
        print("✗ Run  in a terminal manually")

In [ ]:
from agents.agent2 import fetch_url_content, fetch_trend_background

# AGENT 2 NODE
def context_researcher_node(state: NewsroomState) -> dict:
    """Enriches each story in-place with content + background keys."""
    stories = state["stories"]
    for sid, story in stories.items():
        print(f"Agent 2 Starts For : {sid}")
        story["content"]    = fetch_url_content(story["url"])
        story["background"] = fetch_trend_background(story["title"], story["content"])
        print(f"  Agent 2 Ends for: {story['title']}")
        print("=" * 80)
    return {"stories": stories}

In [ ]:
# Run Agent 2 (enriches call['stories'] in-place)
call2 = context_researcher_node(call)

In [ ]:
# Run this in notebook after Agent 2 completes
first = next(iter(call2['stories'].values()))
print("Keys after Agent 2:", list(first.keys()))

In [ ]:
# ── INSPECT after Agent 2 ────────────────────────────────────────────
# Agent 1 fields + Agent 2 new fields: content, background
from urllib.parse import urlparse

# Agent 1 block (unchanged)
print('── Agent 1 fields ──')
print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call2['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

# Agent 2 block (new fields only)
print()
print('── Agent 2 new fields ──')
print(f"{'VEL':>7} {'CONTENT':>9} {'BG':>6}  {'BG_STATUS':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call2['stories'].items():
    vel    = story.get('velocity', 0)
    clen   = len(story.get('content', ''))
    bglen  = len(story.get('background', ''))
    domain = urlparse(story.get('url','')).netloc.replace('www.','')
    title  = story.get('title','')[:45]
    bg_st  = '✅ rich' if bglen > 400 else ('⚠️ thin' if bglen > 0 else '❌ empty')
    print(f"  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {bg_st:<10} {domain:<28} {title}")

---
## Agent 3: Fact Checker
Scores each story's credibility on a **-1 to +1 scale**.
Zero is the natural boundary — negative = discard, positive = keep.

| Signal | Base Weight | How |
|--------|-------------|-----|
| `source_score` | 20% | Domain trust map (32 domains, 0.0 to +0.95) |
| `llm_credibility_check` | 60% | gpt-oss-120b → REAL +0.9 / OPINION +0.1 / SPAM -0.7 |
| `cross_verify` | 20% | Exa → DDG → confirms or contradicts via second source |

**Dynamic reweighting** — weights shift when cross-verify fires:
- Contradiction detected → llm 60%→30%, verify 20%→50%
- Confirmation detected  → llm 60%→50%, verify 20%→30%
- Neutral (not found)    → standard weights unchanged

**Key design decisions:**
- `-1 to +1` range — negative range earned by SPAM or credible contradiction
- Threshold = 0.0 — zero is the natural semantic boundary
- SPAM → -0.7 (only genuine negative LLM signal)
- Empty response / Groq failure → 0.0 neutral (never discard on crash)
- Thin content < 500 chars → 0.0 neutral
- Soft discard — story marked `discarded=True`, never deleted (audit trail)
- Cross-verify: Exa (semantic, HN-aware) → DDG fallback → neutral
- Contradiction check uses `compound-mini` (separate quota from 120b)

In [ ]:
import time                                    # ← add this line
from agents.agent3 import (
    source_score,
    llm_credibility_check,
    check_credibility,
    cross_verify,
)
print("✅ Agent 3 imported")

In [ ]:
# AGENT 3 NODE
def fact_checker_node(state: NewsroomState) -> dict:
    stories = state["stories"]
    for sid, story in stories.items():
        check_credibility(story)
        flag    = "🗑️ DISCARD" if story["discarded"] else "✅ KEEP"
        regime  = story.get("_cred_regime", "?")
        vby     = story.get("verified_by", "NONE")
        contra  = "❌ contradicted" if story.get("contradicted") else ""
        print(f"{story['credibility_score']:+.2f} {flag} "
              f"[{regime}] verified={vby} {contra}| {story['title']}")
        time.sleep(1)
    return {"stories": stories}

In [ ]:
# Run Agent 3 on Agent 2's output (no re-run of A1 or A2)
print("── CREDIBILITY RESULTS ─────────────────────────────")
call3 = fact_checker_node(call2)
print("────────────────────────────────────────────────────")

In [ ]:
# Run this after Agent 3 completes
first = next(iter(call3['stories'].values()))
print("Keys after Agent 3:", list(first.keys()))

In [ ]:
for sid, story in call3["stories"].items():
    print(sid)
    print(story)

In [ ]:
# ── INSPECT after Agent 3 — cumulative (A1 + A2 + A3 fields) ────────
# Keys added: credibility_score, verified_by, contradicted,
#             discarded, _cred_regime, _weights_used
from urllib.parse import urlparse

print('── Agent 1 fields ──')
print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call3['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

print()
print('── Agent 2 new fields ──')
print(f"{'VEL':>7} {'CONTENT':>9} {'BG':>6}  {'BG_STATUS':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call3['stories'].items():
    vel    = story.get('velocity', 0)
    clen   = len(story.get('content', ''))
    bglen  = len(story.get('background', ''))
    domain = urlparse(story.get('url','')).netloc.replace('www.','')
    title  = story.get('title','')[:45]
    bg_st  = '✅ rich' if bglen > 400 else ('⚠️ thin' if bglen > 0 else '❌ empty')
    print(f"  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {bg_st:<10} {domain:<28} {title}")

print()
print('── Agent 3 new fields ──')
print(f"{'FLAG':<4} {'SCORE':<7} {'VEL':>6} {'REGIME':<14} {'WEIGHTS':<12} "
      f"{'VERIFIED_BY':<22} {'SOURCE_DOMAIN':<25} TITLE")
print('-' * 120)
for sid, story in call3['stories'].items():
    flag    = '🗑️' if story.get('discarded') else '✅'
    score   = story.get('credibility_score', 0)
    vel     = story.get('velocity', 0)
    regime  = story.get('_cred_regime', '?')
    weights = story.get('_weights_used', '?')
    vby     = story.get('verified_by', 'NONE')
    contra  = '❌ ' if story.get('contradicted') else ''
    domain  = urlparse(story.get('url','')).netloc.replace('www.','')
    title   = story.get('title','')[:40]
    print(f"{flag}  {score:+.2f}  {vel:>6.1f}  {regime:<14} {weights:<12} "
          f"{contra}{vby:<22} {domain:<25} {title}")


In [ ]:
from urllib.parse import urlparse
# ── INSPECT: full story details after all 3 agents ──────────────────
print(f"{'FLAG':<4} {'SCORE':<7} {'VEL':>6} {'CONTENT':>8} {'BG':>5} {'REGIME':<14} {'VERIFIED_BY':<19} {'ORIGINAL SOURCE':<25} {'TITLE':<50}")
print("-" * 145)



for sid, story in call3["stories"].items():
    flag    = "🗑️" if story.get("discarded") else "✅"
    score   = story.get("credibility_score", 0)
    vel     = story.get("velocity", 0)
    clen    = len(story.get("content", ""))
    bglen   = len(story.get("background", ""))
    regime  = story.get("_cred_regime", "?")
    vby     = story.get("verified_by", "NONE")
    contra  = "❌" if story.get("contradicted") else ""
    #weights = story.get("_weights_used", "?")
    title   = story.get("title", "")
    source = story.get("url", "")
    source_domain = urlparse(
            source
        ).netloc.replace("www.", "")


    print(    f"{flag}  {score:+.2f}  {vel:>6.1f}  {clen:>6}c  "
            f"{bglen:>4}c  {regime:<14} {contra}{vby:<20}"
            f"{source_domain:<25} {title:<50}")

### Save story : till agent 3 (quality data progress saved to disk) so even if some changes comes in agent 1 to agent 3 , agent 4+ code developement will not be affect

In [ ]:

# Save to disk — skip re-processing next time
#from agent_tools.story_cache import save_stories
#save_stories(call3["stories"])

# TO track log of particular function after a hit of particular no. of times pop-up will come


In [ ]:
# importing story cache
# Instead of running A1 → A2 → A3 every time:

from agent_tools.story_cache import load_stories

# Load the saved run (read only — never write again)
cached_stories = load_stories()
print(f"Loaded {len(cached_stories)} stories from cache")


---
## Agent 4: Editorial

Selects the top stories to cover today from Agent 3's credibility-scored pool.

**Responsibilities:**
1. **Filter** — remove Agent 3 discards (credibility_score < 0.0)
2. **Score** — editorial_score = cred×0.50 + vel_norm×0.30 + bg_norm×0.20
3. **Deduplicate** — phi3.5 clusters titles by topic (one call, no bleed)
4. **Select** — top 3 by editorial_score with topic diversity enforced
5. **Route** — ≥1 story → Agent 5 · 0 stories → end + macOS notification

**First conditional edge in the pipeline** — all previous agents were linear.

In [ ]:
# feed directly into Agent 4
from agents.agent4 import editorial_node, route_after_editorial
call4 = editorial_node({"stories": cached_stories})
route = route_after_editorial(call4)
print(f"\nRoute: {route}")

In [ ]:
# ── INSPECT after Agent 4 — cumulative (A1 + A2 + A3 + A4 fields) ───
# Keys added: editorial_score, selected, selection_rank,
#             selection_reason, _vel_norm, _bg_norm,
#             _topic_cluster, _is_duplicate
from urllib.parse import urlparse

print('── Agent 1 fields ──')
print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call4['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

print()
print('── Agent 2 new fields ──')
print(f"{'VEL':>7} {'CONTENT':>9} {'BG':>6}  {'BG_STATUS':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call4['stories'].items():
    vel    = story.get('velocity', 0)
    clen   = len(story.get('content', ''))
    bglen  = len(story.get('background', ''))
    domain = urlparse(story.get('url','')).netloc.replace('www.','')
    title  = story.get('title','')[:45]
    bg_st  = '✅ rich' if bglen > 400 else ('⚠️ thin' if bglen > 0 else '❌ empty')
    print(f"  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {bg_st:<10} {domain:<28} {title}")

print()
print('── Agent 3 new fields ──')
print(f"{'FLAG':<4} {'SCORE':<7} {'VEL':>6} {'REGIME':<14} {'WEIGHTS':<12} "
      f"{'VERIFIED_BY':<22} {'SOURCE_DOMAIN':<25} TITLE")
print('-' * 120)
for sid, story in call4['stories'].items():
    flag    = '🗑️' if story.get('discarded') else '✅'
    score   = story.get('credibility_score', 0)
    vel     = story.get('velocity', 0)
    regime  = story.get('_cred_regime', '?')
    weights = story.get('_weights_used', '?')
    vby     = story.get('verified_by', 'NONE')
    contra  = '❌ ' if story.get('contradicted') else ''
    domain  = urlparse(story.get('url','')).netloc.replace('www.','')
    title   = story.get('title','')[:40]
    print(f"{flag}  {score:+.2f}  {vel:>6.1f}  {regime:<14} {weights:<12} "
          f"{contra}{vby:<22} {domain:<25} {title}")

print()
print('── Agent 4 new fields ──')
print(f"{'SEL':<5} {'RANK':<5} {'ED_SCORE':<10} {'VEL_N':<7} {'BG_N':<7} "
      f"{'CLUSTER':<9} {'IS_DUPLICATE':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 110)
for sid, story in call4['stories'].items():
    selected  = '✅' if story.get('selected') else '—'
    rank      = story.get('selection_rank', '-')
    ed_score  = story.get('editorial_score', 0)
    vel_n     = story.get('_vel_norm', 0)
    bg_n      = story.get('_bg_norm', 0)
    cluster   = story.get('_topic_cluster', '-')
    is_dup    = '❌' if story.get('_is_duplicate') else '✅'
    domain    = urlparse(story.get('url','')).netloc.replace('www.','')
    title     = story.get('title','')[:40]
    print(f"{selected:<5} #{rank!s:<4} {ed_score:<10.3f} {vel_n:<7.3f} {bg_n:<7.3f} "
          f"{cluster!s:<9} {is_dup:<10} {domain:<28} {title}")

---
## Agent 5: Script Writer
Converts the top 3 selected stories into a 60-90 second YouTube Shorts script.

**Model:** llama-3.3-70b-versatile (Groq) — separate quota from Agent 3
**Format:** HOOK → S1_CONTEXT → S1_CORE → S1_TWIST → S2_HOOK → ... → CTA
**Word target:** 150-225 words (~60-90 seconds at spoken pace)
**Tone calibration:** credibility_score from Agent 3 controls framing confidence

In [ ]:
from agents.agent5 import script_writer_node

# Run Agent 5 on Agent 4's output
call5 = script_writer_node(call4)

# Print the full script
print("\n── FULL SCRIPT ──────────────────────────────────────────────────")
print(call5["script"]["full_text"])
print(f"\n── STATS ────────────────────────────────────────────────────────")
print(f"Words:    {call5['script']['word_count']}")
print(f"Duration: ~{call5['script']['est_duration']}")
print(f"Sections: {len(call5['script']['sections'])}")
print(f"Stories:  ranks {call5['script']['stories_used']}")
print(f"Attempts: {call5['script']['attempt']}")

---
## 🗂️ Cache & Checkpoint Reference

### `story_cache.py` — A1+A2+A3 output
Reload to test Agent 4 or Agent 5 without re-running A1-A3.
```python
from agent_tools.story_cache import load_stories
stories = load_stories()
call4 = editorial_node({"stories": stories})
```
File: `data/stories_cache.json`

### `pipeline_cache.py` — A1+A2+A3+A4+A5 output
Reload to test Agent 6+ without re-running A1-A5.
```python
from agent_tools.pipeline_cache import load_checkpoint
state = load_checkpoint("till-agent5")
call6 = script_qc_node(state)
```
File: `data/checkpoints/till-agent5.json`

| Working on... | Load this | Contains |
|---|---|---|
| Agent 4 or 5 | `load_stories()` | stories only |
| Agent 6+ | `load_checkpoint("till-agent5")` | stories + script |

---

In [ ]:
# ── Save checkpoint after Agent 5 ────────────────────────────────────
from agent_tools.pipeline_cache import save_checkpoint, list_checkpoints

save_checkpoint(call5, name="till-agent5")
print(f"Available checkpoints: {list_checkpoints()}")

---
## Agent 6: Script QC + Agent 6.1: Voice-Over Generator

Two separate agents with a clean division of labor — text correctness
vs. audio generation reliability.

### Agent 6 — Script QC (text intelligence)
Validates and polishes Agent 5's script until it's genuinely ready to
become audio. Two-stage model split, mirrors Agent 3's design:

| Stage | Model | Job |
|---|---|---|
| JUDGE | `gpt-oss-120b` | Finds problems: word count, AI-voice phrasing, weak transitions, restated twists, CTA format |
| REWRITE | `llama-3.3-70b-versatile` | Fixes ONLY flagged sections — never regenerates the whole script |

**Also handles (pure Python, no LLM cost):**
- Date humanization ("August 23, 2024" → "last year") — uses the real
  current date, since Agent 5 has no clock
- Deterministic `annotated_text` / `tts_ready_text` split — pacing
  markers are derived from sentence structure, never generated by an
  LLM, and **never sent to Kokoro** (Kokoro can't render `[PAUSE]`/
  `[EMPHASIS]` tags — see KNOWN_ISSUES ISSUE-13)

**Control:** APPROVE/REVISE loop, max 2 iterations. Never blocks the
pipeline — approves with a logged warning if issues remain after 2 tries.

### Agent 6.1 — Voice-Over Generator (audio reliability)
Converts Agent 6's QC'd `tts_ready_text` into actual voice-over audio
using **Kokoro** (via `mlx-audio`, Apple Metal GPU acceleration, $0 cost,
fully local — see KNOWN_ISSUES ISSUE-12).

Not in the original 10-agent roadmap — added after Agent 1 itself
surfaced the Kokoro HN story, Agent 3 confirmed it, and Agent 4 selected
it for scripting. Kept **separate** from Agent 6 because text QC and
audio generation are genuinely different concerns with different
failure modes — Agent 6.1 could later swap TTS backends without
touching Agent 6 at all.

**Reliability guarantee, not just "did the command exit 0":**
generates audio, then verifies actual duration roughly matches the
word-count estimate (±50%) — catches garbled or truncated speech that
a bare success/failure check would miss. Only runs if Agent 6 approved
the script — trusts Agent 6's QC gate completely.

In [ ]:
from agent_tools.pipeline_cache import load_checkpoint
from agents.agent6 import script_qc_node

state = load_checkpoint("till-agent5")
call6 = script_qc_node(state)

print("\n── ANNOTATED TEXT ──────────────────────────────────")
print(call6["script"]["annotated_text"])
print("\n── TTS-READY TEXT ──────────────────────────────────")
print(call6["script"]["tts_ready_text"])
print(f"\nApproved: {call6['script']['approved']}")
print(f"Iterations: {call6['script']['iterations']}")
print(f"QC notes: {call6['script']['qc_notes']}")

In [ ]:
from agents.agent61 import 